# Etapa 4 — Regressão: externas × domésticas

$$\Delta\ln(\text{USD/BRL})_t = \alpha + \beta_1 X_t + \beta_2 B_t + \varepsilon_t$$

- **Três modelos**: só externas, só domésticas, completo.
- Cada um **com e sem** dummy para abr/2020 (outlier do primário).
- Variáveis **padronizadas** (z-score) → β = efeito de um choque de 1 desvio-padrão, em desvios-padrão do câmbio.
- Erros-padrão **HAC (Newey-West)** com 4 defasagens.

In [1]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey

dados = pd.read_csv("dados/tratados/variacoes_mensais.csv", index_col=0, parse_dates=True)

EXTERNAS = ["d_ust10", "dl_vix", "dl_dolar", "dl_commod"]
DOMESTICAS = ["d_selic", "dl_cds", "d_primario"]
Y = "dl_cambio"
LAGS_HAC = 4   # regra de bolso: 0,75 * T^(1/3), com T = 214

## 1. Padronização

z = (x − média) / desvio-padrão. Depois disso toda variável tem média 0 e desvio-padrão 1.
A dummy **não** é padronizada: ela vale 1 em abr/2020 e 0 nos demais meses.

In [2]:
z = (dados - dados.mean()) / dados.std()
z["dummy_abr2020"] = (z.index == "2020-04-30").astype(int)

z.describe().loc[["mean", "std"]].round(2)   # conferência: médias 0 e desvios 1

,dl_cambio,d_ust10,dl_vix,dl_dolar,dl_commod,d_selic,dl_cds,d_primario,dummy_abr2020
mean,0.0,0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,0.00
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.07


## 2. Correlação entre os regressores

Antes de regredir, veja como os regressores se relacionam. Correlação alta entre uma externa e uma
doméstica = **sobreposição** de R² nos modelos separados (e o motivo da etapa 5).

In [3]:
z[EXTERNAS + DOMESTICAS + [Y]].corr().round(2)

,d_ust10,dl_vix,dl_dolar,dl_commod,d_selic,dl_cds,d_primario,dl_cambio
d_ust10,1.00,-0.05,0.12,0.23,0.05,-0.03,0.05,-0.08
dl_vix,-0.05,1.00,0.39,-0.29,-0.04,0.40,0.12,0.29
dl_dolar,0.12,0.39,1.00,-0.54,0.05,0.55,0.05,0.59
dl_commod,0.23,-0.29,-0.54,1.00,0.11,-0.47,0.10,-0.57
d_selic,0.05,-0.04,0.05,0.11,1.00,0.11,0.09,-0.02
dl_cds,-0.03,0.40,0.55,-0.47,0.11,1.00,-0.10,0.71
d_primario,0.05,0.12,0.05,0.10,0.09,-0.10,1.00,-0.09
dl_cambio,-0.08,0.29,0.59,-0.57,-0.02,0.71,-0.09,1.00


## 3. Estimação

`cov_type="HAC"` troca **só** a matriz de variância dos β (erros-padrão). Coeficientes e R² são os do MQO comum.

In [4]:
def estimar(regressores, com_dummy):
    X = z[regressores + (["dummy_abr2020"] if com_dummy else [])]
    X = sm.add_constant(X)
    return sm.OLS(z[Y], X).fit(cov_type="HAC", cov_kwds={"maxlags": LAGS_HAC})

especificacoes = {"(1) Externas": EXTERNAS, "(2) Domésticas": DOMESTICAS, "(3) Completo": EXTERNAS + DOMESTICAS}

modelos = {}
for com_dummy in [False, True]:
    for nome, regs in especificacoes.items():
        modelos[(nome, "com dummy" if com_dummy else "sem dummy")] = estimar(regs, com_dummy)

## 4. Tabela de resultados

Cada célula: coeficiente padronizado e estrelas de significância
(\*\*\* p<0,01; \*\* p<0,05; \* p<0,10), com p-valores HAC.

In [5]:
def estrelas(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

linhas = EXTERNAS + DOMESTICAS + ["dummy_abr2020"]
tabela = pd.DataFrame(index=linhas + ["R²", "R² ajustado", "N"])

for chave, res in modelos.items():
    coluna = {}
    for v in linhas:
        coluna[v] = f"{res.params[v]:.3f}{estrelas(res.pvalues[v])}" if v in res.params else ""
    coluna["R²"] = f"{res.rsquared:.3f}"
    coluna["R² ajustado"] = f"{res.rsquared_adj:.3f}"
    coluna["N"] = int(res.nobs)
    tabela[chave] = pd.Series(coluna)

tabela.columns = pd.MultiIndex.from_tuples(tabela.columns)
tabela

,(1) Externas,(2) Domésticas,(3) Completo,(1) Externas,(2) Domésticas,(3) Completo
,sem dummy,sem dummy,sem dummy,com dummy,com dummy,com dummy
d_ust10,-0.048,,-0.044,-0.047,,-0.043
dl_vix,0.031,,-0.074,0.041,,-0.073
dl_dolar,0.407***,,0.229***,0.406***,,0.225***
dl_commod,-0.329***,,-0.200***,-0.325***,,-0.203***
d_selic,,-0.102**,-0.072,,-0.106**,-0.074
dl_cds,,0.724***,0.529***,,0.726***,0.530***
d_primario,,-0.006,-0.008,,0.029,0.024
dummy_abr2020,,,,0.965***,0.612,0.559
R²,0.440,0.520,0.610,0.444,0.521,0.611


## 5. Decomposição do R²

| Componente | Fórmula | Significado |
|---|---|---|
| Só externo | R²(3) − R²(2) | o que as externas acrescentam **além** das domésticas |
| Só doméstico | R²(3) − R²(1) | o que as domésticas acrescentam **além** das externas |
| Sobreposição | R²(1) + R²(2) − R²(3) | parte que **ambos os blocos** "explicam" — não dá para atribuir a nenhum |

As três partes somam R²(3).

In [6]:
decomp = {}
for versao in ["sem dummy", "com dummy"]:
    r2_ext = modelos[("(1) Externas", versao)].rsquared
    r2_dom = modelos[("(2) Domésticas", versao)].rsquared
    r2_tot = modelos[("(3) Completo", versao)].rsquared
    decomp[versao] = {
        "R² externas (1)": r2_ext,
        "R² domésticas (2)": r2_dom,
        "R² completo (3)": r2_tot,
        "— só externo": r2_tot - r2_dom,
        "— só doméstico": r2_tot - r2_ext,
        "— sobreposição": r2_ext + r2_dom - r2_tot,
    }
pd.DataFrame(decomp).round(3)

,sem dummy,com dummy
R² externas (1),0.44,0.444
R² domésticas (2),0.52,0.521
R² completo (3),0.61,0.611
— só externo,0.09,0.090
— só doméstico,0.17,0.166
— sobreposição,0.35,0.354


## 6. Diagnóstico dos resíduos (modelo completo, sem dummy)

Os testes do Gujarati que justificam (ou não) o HAC:
- **Breusch-Pagan** — H₀: homocedasticidade.
- **Breusch-Godfrey** (4 defasagens) — H₀: sem autocorrelação.

p < 0,05 → rejeita H₀ → o erro-padrão clássico estaria errado.

In [7]:
completo = modelos[("(3) Completo", "sem dummy")]

p_bp = het_breuschpagan(completo.resid, completo.model.exog)[1]
p_bg = acorr_breusch_godfrey(completo, nlags=4, result_object=False)[1]
print(f"Breusch-Pagan   p = {p_bp:.3f}")
print(f"Breusch-Godfrey p = {p_bg:.3f}")

# Comparação direta: erros-padrão clássicos x HAC (mesmos β)
classico = sm.OLS(completo.model.endog, completo.model.exog).fit()
pd.DataFrame({
    "β": completo.params.values,
    "EP clássico": classico.bse,
    "EP HAC": completo.bse.values,
}, index=completo.params.index).round(3)

Breusch-Pagan   p = 0.286
Breusch-Godfrey p = 0.090


,β,EP clássico,EP HAC
const,0.000,0.043,0.037
d_ust10,-0.044,0.047,0.053
dl_vix,-0.074,0.050,0.059
dl_dolar,0.229,0.061,0.057
dl_commod,-0.200,0.058,0.061
d_selic,-0.072,0.045,0.046
dl_cds,0.529,0.057,0.071
d_primario,-0.008,0.045,0.034


## 7. Saída completa do statsmodels (modelo completo, sem dummy)

O mesmo formato de tabela que aparece nos livros — bom para se acostumar a ler.

In [8]:
print(completo.summary())

                            OLS Regression Results                            
Dep. Variable:              dl_cambio   R-squared:                       0.610
Model:                            OLS   Adj. R-squared:                  0.597
Method:                 Least Squares   F-statistic:                     34.17
Date:                Sun, 08 Nov 2026   Prob (F-statistic):           2.46e-31
Time:                        15:49:17   Log-Likelihood:                -202.28
No. Observations:                 214   AIC:                             420.6
Df Residuals:                     206   BIC:                             447.5
Df Model:                           7                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       6.939e-18      0.037   1.87e-16      1.0

In [9]:
tabela.to_csv("resultados/etapa4_regressoes.csv")
pd.DataFrame(decomp).to_csv("resultados/etapa4_decomposicao_r2.csv")